# Exp1 evaluation: application-only vs multi-source models

Compares models trained on different data sources (e.g. App, +Bureau, +Previous, +Repayment, All sources) on the same 15,000 test applicants.

**How to use**
1. Train your models on `split == 'train'` and tune on `split == 'val'` (from `clean_data/splits.csv`).
2. Predict a score for **every val and test applicant** with each model.
3. Put them in `MODELS` below and **Run All**.

If `MODELS` is left empty, the notebook runs a demo with two dummy models so you can see the output format.

In [1]:
import sys
from pathlib import Path
import pandas as pd

HERE = next(p for p in [Path.cwd(), Path.cwd() / "outputs" / "exp1"] if (p / "eval_exp1.py").exists())
sys.path.insert(0, str(HERE))
sys.dont_write_bytecode = True
import eval_exp1 as ev

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Your models

In [2]:
# Fill in: model name -> predictions for EVERY val and test applicant, as either
#   - a DataFrame with columns SK_ID_CURR, score   (score = predicted probability of default), or
#   - the path to a CSV file with those two columns.
# The FIRST model is the baseline that the others are compared against.
MODELS = {
    # "App only": app_only_df,
    # "All sources": "path/to/all_sources.csv",
}
N_BOOT = 1000   # bootstrap resamples for the 95% confidence intervals (200 is faster for a quick look)
SAVE = True     # also save the result tables as CSV files in this folder

In [3]:
DEMO = not MODELS
if DEMO:
    # Two deliberately simple logistic regressions, trained on train only, so you can see what the output looks like.
    from sklearn.impute import SimpleImputer
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    app = ev.attach_split(pd.read_parquet(ev.CLEAN_DIR / "application.parquet",
                                          columns=["SK_ID_CURR", "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "DAYS_BIRTH"]))
    for name, cols in [("Demo small", ["EXT_SOURCE_2", "DAYS_BIRTH"]),
                       ("Demo bigger", ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "DAYS_BIRTH"])]:
        m = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression())
        m.fit(app.loc[app.split == "train", cols], app.loc[app.split == "train", "TARGET"])
        scored = app[app.split.isin(["val", "test"])]
        MODELS[name] = pd.DataFrame({"SK_ID_CURR": scored.SK_ID_CURR, "score": m.predict_proba(scored[cols])[:, 1]})
    print("MODELS was empty, so this is a DEMO with two dummy models. These numbers are NOT results.")

MODELS was empty, so this is a DEMO with two dummy models. These numbers are NOT results.


## 2. Results

In [4]:
table, comps, preds = ev.compare_prediction_files(list(MODELS.values()), labels=list(MODELS), n_boot=N_BOOT)

### Test-set metrics
One row per model. Each cell shows **value (95% confidence interval)**. The threshold used for precision, recall and
F1 was chosen on the **validation** set (the value that maximises F1), so the test set stays untouched.

| Metric | Meaning |
|---|---|
| `roc_auc` | how well the model ranks defaulters above non-defaulters (0.5 = random, 1 = perfect) |
| `pr_auc` | precision-recall area; more informative with only ~8% defaulters (8% = random) |
| `precision` / `recall` / `f1` | at the chosen threshold: share of flagged who default / share of defaulters flagged / balance of both |
| `precision@top10%` / `recall@top10%` | if the riskiest 10% of applicants are reviewed: share who default / share of all defaulters caught |

In [5]:
ev.pretty_metrics(table)

,threshold,roc_auc,pr_auc,precision,recall,f1,precision@top10%,recall@top10%
model,,,,,,,,
Demo small,0.114,0.653 (0.637-0.670),0.142 (0.130-0.156),0.155 (0.143-0.169),0.374 (0.349-0.401),0.219 (0.204-0.236),0.181 (0.161-0.199),0.224 (0.202-0.246)
Demo bigger,0.147,0.710 (0.695-0.725),0.183 (0.166-0.203),0.204 (0.186-0.223),0.323 (0.296-0.348),0.250 (0.230-0.270),0.221 (0.199-0.241),0.274 (0.249-0.295)


### Is each model better than the baseline?
Same test applicants for every model, so the comparison is paired. **Significantly better** means the 95% confidence
interval of the difference lies entirely above 0. The p-value comes from the DeLong test for ROC-AUC and from the
paired bootstrap for PR-AUC.

In [6]:
ev.pretty_comparisons(comps)

,model,vs_baseline,metric,difference (95% CI),p_value,verdict
0,Demo bigger,Demo small,roc_auc,+0.0566 (+0.0445 to +0.0690),2.8e-20,significantly better
1,Demo bigger,Demo small,pr_auc,+0.0412 (+0.0295 to +0.0534),0,significantly better


## 3. Save

In [7]:
if SAVE and not DEMO:
    table.to_csv(HERE / "exp1_metrics.csv")              # numeric values incl. CI bounds
    comps.to_csv(HERE / "exp1_comparisons.csv", index=False)
    print("Saved result tables in", HERE)
else:
    print("Not saved (demo mode or SAVE = False).")

Not saved (demo mode or SAVE = False).
